# 16 - Louvain, 3 hospitals: FINAL protocol (per-hospital cutoffs, FedAvg + fine-tune, topo v3)

**This is meant to be the last training run on this split.** Design and win rule are fixed in
`docs/2026-10-28-final-protocol-preregistration.md` (read it first). Split: `louvain_split(n_clients=3)` (3 hospitals; seeds 42/46 and 43/45 give the same partition).

What it runs (rounds = 8, local epochs = 4, local-only epochs = 32, hidden = 128, seeds 42-46):

| arm | note |
|---|---|
| FedAvg | also trains **local-only, pooled and FedAvg + fine-tune** once per seed |
| v3 at T = 0.25 / 0.5 / 1.0 | every hospital gets its own similarity-weighted model (`PersonalizedStrategy`) |
| v3-shuffled | v3 at the T chosen on **validation**; fingerprints given to the wrong hospitals |
| Uniform, Topo v1 | the topology baseline and its uniform reference |
| FedProx, FedMedian | standard baselines |

**Primary metric:** test macro-F1 where every hospital picks its **own cutoff** on its own validation rows and
the round is chosen on validation; test is read once. **Everything needed later is saved** (confusion counts per
cutoff, probability histograms, weights, fingerprints, partitions, provenance), so no rerun should be needed.

Results: `results/amr_louvain_final_<TAG>.json` (checkpointed after every arm and resumed if present; on Colab also mirrored to Drive).
Not run on purpose: Topo-shuffled / Inverse-size (notebook 15 already answered them; v3-shuffled is the fingerprint test now) and FedTrimmedAvg (equals Uniform).

In [ ]:
# 1) Environment: Colab (clone + install + mount Drive) or local (.venv)
import os, sys
ON_COLAB = 'google.colab' in sys.modules
BRANCH = 'feat/federated-baselines'
if ON_COLAB:
    !git clone -b main https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout main && git pull)
    !cd /content/Capstone-amr-fed && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
    !cd /content/Capstone-amr-fed && git log -1 --oneline
    !pip install -q torch_geometric 'flwr[simulation]' networkx scipy
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # edit to your folder if different
else:
    os.environ.setdefault('ARMD_DIR', '/Users/vikramadityasharma/Library/CloudStorage/GoogleDrive-vikramaditya.raunak@gmail.com/.shortcut-targets-by-id/1uVx5PmldPyJBMtnB9Lkl9N1rNBQrmJt-/ARMD')
import torch, ray, flwr, torch_geometric
print('colab:', ON_COLAB, '| python', sys.version.split()[0], '| torch', torch.__version__, '| flwr', flwr.__version__,
      '| ray', ray.__version__, '| device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('ARMD_DIR:', os.environ['ARMD_DIR'])

In [ ]:
# 2) Sanity: data resolves + this is the final-protocol branch (modules below must exist)
from pathlib import Path
if ON_COLAB:
    SRC = Path('/content/Capstone-amr-fed/src')
else:
    SRC = next(p / 'src' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'amr_fed').is_dir())
sys.path.insert(0, str(SRC))
from amr_fed import config
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong - fix cell 1 and re-run.'
from amr_fed.federated import final_protocol as P, analysis as A, selection
from amr_fed.topology import personalized_weights, train_frame
print('imports OK | v3 temperatures:', P.T_GRID, '| fine-tune epochs:', P.FT_EPOCHS)

In [ ]:
# 3) Load cohort once + smoke (expect 3 hospitals per seed; seeds 42/46 and 43/45 share a partition)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import louvain_split
df = load_cohort_frame()
print('rows:', f'{len(df):,}')
SEEDS = (42, 43, 44, 45, 46)
for s in SEEDS:
    a = louvain_split(df, n_clients=3, seed=s)
    print(f'seed {s}: {a.nunique()} hospitals | patients (largest first): {a.value_counts().sort_values(ascending=False).tolist()}')
    assert a.index.is_unique and len(a) == df[PK].nunique(), 'not 1:1'
print('SMOKE PASS')

In [ ]:
# 4) All arms (resume-safe). Longest cell: 9 federated runs per seed x 5 seeds + local-only/pooled/fine-tune once per seed.
import os, ray, json
os.environ.pop('AMR_RNG_PROBE', None)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed') and not m.startswith('amr_fed.federated.final')]:
    del _s.modules[_m]
from amr_fed.federated import final_protocol as P, analysis as A
from amr_fed.partition import louvain_split
TAG = 'all'
KW = dict(seeds=SEEDS, df=df, partition_fn=lambda d, s: louvain_split(d, n_clients=3, seed=s),
          n_clients=3, rounds=8, local_epochs=4, local_only_epochs=32, hidden=128)
RESULTS = SRC.parent / 'results' / f'amr_louvain_final_{TAG}.json'
RESULTS.parent.mkdir(exist_ok=True)
MIRROR = Path(f'/content/drive/MyDrive/amr_louvain_final_{TAG}.json') if ON_COLAB else None
src = RESULTS if RESULTS.exists() else (MIRROR if MIRROR and MIRROR.exists() else None)
res = json.load(open(src)) if src else {}
res = P.run_all(res, prefix='louvain', include_trimmed=False, kw=KW,
                checkpoint=lambda r: P.save_json(r, RESULTS, MIRROR), ray_shutdown=ray.shutdown)
P.save_json(res, RESULTS, MIRROR)
print('arms done:', [k for k in res if not k.startswith('_')])

In [ ]:
# 5) Summary: primary metric (per-hospital cutoffs), shared-cutoff F1, worst hospital, AUROC, chosen round per seed
import json, numpy as np
print(A.summary_table(res))
print('\nchosen v3 temperature (validation only):', res['_meta'].get('chosen_T'), '| mean val F1 by T:', res['_meta'].get('T_val_scores'))
print('wall-clock per arm (min, mean over seeds):', {k: round(np.mean([r["wall_seconds"] for r in v["runs"]]) / 60, 1) for k, v in res.items() if not k.startswith('_')})

In [ ]:
# 6) Paired comparisons (by seed) + the pre-registered verdict
print(A.comparison_report(res))
print(A.verdict(res))

In [ ]:
# 7) Per-hospital test F1 (mean over seeds, each hospital at its own cutoff)
print(A.per_hospital_table(res, rank_by_size=True))

In [ ]:
# 8) Weights the topology / v3 arms used (self-weight = how much a hospital keeps its own model)
print(A.weights_report(res))